In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
# ── Eq.(4): analytical mean-field solutions ──────────────────────────────────
def rho_plus(lam, lam_D):
    if lam_D < 1e-9: lam_D = 1e-9
    disc = (lam - lam_D)**2 - 4*lam_D*(1 - lam)
    if disc < 0: return np.nan
    val = (lam_D - lam + np.sqrt(disc)) / (2*lam_D)
    return val if val > 0 else np.nan

def rho_minus(lam, lam_D):
    if lam_D < 1e-9: lam_D = 1e-9
    disc = (lam - lam_D)**2 - 4*lam_D*(1 - lam)
    if disc < 0: return np.nan
    rp = rho_plus(lam, lam_D)
    if rp is None or np.isnan(rp): return np.nan
    val = (lam_D - lam - np.sqrt(disc)) / (2*lam_D)
    return val if (val > 0 and val < rp) else np.nan

def jump_point(lam_D):
    """
    At lambda_c the discriminant is mathematically 0,
    but floating-point gives ~1e-15. Use max(disc,0) to fix.
    rho at jump = (lD - lc) / (2*lD)  [from disc=0 formula]
    """
    if lam_D <= 1: return None, None
    lc   = 2*np.sqrt(lam_D) - lam_D
    if lc <= 0: return None, None
    rho_j = (lam_D - lc) / (2*lam_D)   # exact formula when disc=0
    return lc, rho_j


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]

lam_vals   = np.linspace(0.05, 2.2, 600)
lam_D_list = [0.0, 0.3, 0.6, 1.0, 1.5, 2.0, 2.5, 3.0]
colors     = plt.cm.viridis(np.linspace(0.15, 0.85, len(lam_D_list)))

for lam_D, color in zip(lam_D_list, colors):
    eff = max(lam_D, 1e-9)
    rp  = np.array([rho_plus (l, eff) for l in lam_vals], dtype=float)
    rm  = np.array([rho_minus(l, eff) for l in lam_vals], dtype=float)

    ax.plot(lam_vals, rp, '-',  color=color, lw=1.8,
            label=r'$\lambda_\Delta=%.1f$' % lam_D)
    ax.plot(lam_vals, rm, '--', color=color, lw=1.4)

    if lam_D > 1:
        # 只画 λ <= 1 的部分（因为 λ=1 之后 0 失稳）
        lam_zero = np.linspace(0, 1.0, 100)
        rho_zero = np.zeros_like(lam_zero)
        ax.plot(lam_zero, rho_zero, '-', color=color, lw=1.8,
                alpha=1)

    lc, rho_j = jump_point(lam_D)
    if lc is not None:
        # hollow circle at top of jump (on stable branch)
        ax.plot(lc, rho_j, 'o', color=color, ms=6,
                mfc='white', mew=1.5, zorder=5)

ax.axvline(x=1.0, color='gray', lw=1, ls=':', alpha=0.7)
ax.text(1.02, 0.02, r'$\lambda=1$', fontsize=9, color='gray')
ax.set_xlim(0.2, 2.2);  ax.set_ylim(-0.02, 0.85)
ax.spines['bottom'].set_position(('data', -0.01))
ax.set_xlabel(r'Rescaled infectivity $\lambda$', fontsize=13)
ax.set_ylabel(r'Stationary density $\rho^*$',    fontsize=13)
ax.set_title('Fig 4a — Bifurcation diagram', fontsize=12)
ax.legend(fontsize=8, ncol=2, loc='upper left',
          handlelength=1.2, labelspacing=0.2)

In [ ]:
# ── Fig 4b: 2D phase diagram heatmap ─────────────────────────────────────────
ax2 = axes[1]
lg  = np.linspace(0.0, 2.0, 300)
ldg = np.linspace(0.0, 5.0, 300)
LAM, LAM_D = np.meshgrid(lg, ldg)
Z = np.zeros_like(LAM)
for i in range(Z.shape[0]):
    for j in range(Z.shape[1]):
        v = rho_plus(LAM[i,j], LAM_D[i,j])
        Z[i,j] = v if (v is not None and not np.isnan(v)) else 0.0

im = ax2.imshow(Z, origin='lower', aspect='auto',
                extent=[lg[0], lg[-1], ldg[0], ldg[-1]],
                cmap='viridis', vmin=0, vmax=0.8)
plt.colorbar(im, ax=ax2, label=r'$\rho^*_{2+}$')

ax2.axvline(x=1.0, color='white', lw=1.5, ls='--', alpha=0.8)
lDc = np.linspace(1.0, 5.0, 300)
lcc = 2*np.sqrt(lDc) - lDc
mask = (lcc >= 0) & (lcc <= 2.0)
ax2.plot(lcc[mask], lDc[mask], 'w-.', lw=1.8,
         label=r'$\lambda_c=2\sqrt{\lambda_\Delta}-\lambda_\Delta$')
ax2.axhline(y=1.0, color='white', lw=1, ls=':', alpha=0.6)
ax2.set_xlabel(r'Rescaled infectivity $\lambda$',       fontsize=13)
ax2.set_ylabel(r'$\lambda_\Delta$ (triangle infectivity)', fontsize=13)
ax2.set_title('Fig 4b — Phase diagram heatmap', fontsize=12)
ax2.legend(fontsize=9, loc='upper right')
ax2.text(1.03, 0.15, r'$\lambda=1$',      fontsize=9, color='white')
ax2.text(0.05, 1.08, r'$\lambda_\Delta=1$', fontsize=9, color='white')

plt.tight_layout()
plt.savefig('fig4_phase_diagram.png',
            dpi=150, bbox_inches='tight')
print("Saved: fig4_phase_diagram.png")
